# Using analysis project role

Code from Helen

In [6]:
import json
import logging

import boto3
from botocore.exceptions import ClientError
from dateutil import tz

from aind_data_access_api.document_db import MetadataDbClient

# AWS Environment
# ACCOUNT_ID = "225952147762"
ACCOUNT_ID = "467914378000"
DOC_DB_API_HOST = "api.allenneuraldynamics-test.org"

# for all analysis projects
ANALYSIS_ROLE_NAME = "AindAnalysisProjectsRole"
ANALYSIS_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{ANALYSIS_ROLE_NAME}"
ANALYSIS_DB_NAME = "analysis"
# for dynamic-foraging-analysis project
PROJECT = "dynamic-foraging-analysis"
BUCKET = f"aind-{PROJECT}-prod-o5171v"


logging.basicConfig(level=logging.INFO)


def get_session_credentials():
    sts_client = boto3.client("sts")
    # optional
    origin = sts_client.get_caller_identity()
    origin_user = origin["Arn"].split("/")[-1] if "Arn" in origin else "unknown"
    # assume analysis projects role
    role_session_name = f"{ANALYSIS_ROLE_NAME}-session-{origin_user[:12]}"
    resp = sts_client.assume_role(
        RoleArn=ANALYSIS_ROLE_ARN, RoleSessionName=role_session_name
    )
    credentials = resp["Credentials"]
    logging.info(
        f"Assumed role {ANALYSIS_ROLE_NAME}. Session will expire at"
        f"{credentials['Expiration'].astimezone(tz=tz.tzlocal())}."
    )
    return credentials


def create_session(credentials):
    session = boto3.Session(
        aws_access_key_id=credentials["AccessKeyId"],
        aws_secret_access_key=credentials["SecretAccessKey"],
        aws_session_token=credentials["SessionToken"],
    )
    return session


def test_read_write_delete_docdb(session=None):
    record = {"_id": f"test_write_to_docdb"}

    docdb_api_client = MetadataDbClient(
        host=DOC_DB_API_HOST,
        database=ANALYSIS_DB_NAME,
        collection=PROJECT,
        boto_session=session,
    )
    resp_read = docdb_api_client._count_records()
    resp_write = docdb_api_client.upsert_one_docdb_record(record)
    resp_delete = docdb_api_client.delete_one_record(record["_id"])

    logging.info(f"DocDB: read ({resp_read})/ write ({resp_write.status_code})/ delete ({resp_delete.status_code})")
    return


def test_read_write_delete_s3(session=None):
    key = "test/test_write_to_S3.json"
    file = {"_id": "test_write_to_S3"}
    file_json = json.dumps(file, indent=3, sort_keys=True)

    s3_client = (session.client("s3") if session is not None else boto3.client("s3"))
    resp_read = s3_client.list_objects_v2(Bucket=BUCKET)["ResponseMetadata"]["HTTPStatusCode"]
    try:
        resp_write = s3_client.put_object(Bucket=BUCKET, Key=key, Body=file_json)["ResponseMetadata"]["HTTPStatusCode"]
    except ClientError as e:
        resp_write = e.response["Error"]["Code"]
    try:
        resp_delete = s3_client.delete_object(Bucket=BUCKET, Key="test/test_write_to_S3.json")["ResponseMetadata"]["HTTPStatusCode"]
    except ClientError as e:
        resp_delete = e.response["Error"]["Code"]

    logging.info(f"S3: read ({resp_read})/ write ({resp_write})/ delete ({resp_delete})")
    return


if __name__ == "__main__":
    # for default user, only read should work since 
    # analysis db and buckets are public readonly
    test_read_write_delete_docdb()
    test_read_write_delete_s3()

    # read/write/delete should all work after assuming role
    credentials = get_session_credentials()
    session = create_session(credentials)
    test_read_write_delete_docdb(session=session)
    test_read_write_delete_s3(session=session)


INFO:root:DocDB: read ({'total_record_count': 1, 'filtered_record_count': 1})/ write (403)/ delete (403)
INFO:root:S3: read (200)/ write (200)/ delete (204)


ClientError: An error occurred (AccessDenied) when calling the AssumeRole operation: User: arn:aws:sts::467914378000:assumed-role/aind-codeocean-user/mennoiee-pekm-kego-beal-oofdakoddfam@worker is not authorized to perform: sts:AssumeRole on resource: arn:aws:iam::467914378000:role/AindAnalysisProjectsRole

# Get result from docDB via ssh

In [2]:
from aind_data_access_api.document_db_ssh import DocumentDbSSHClient, DocumentDbSSHCredentials

credentials = DocumentDbSSHCredentials()
credentials.database = "behavior_analysis"
credentials.collection = "mle_fitting"

In [3]:
with DocumentDbSSHClient(credentials=credentials) as client:
    response = client.collection.find(limit=1)


2024-11-25 23:59:22,760| ERROR   | Exception (client): Error reading SSH protocol banner[Errno 104] Connection reset by peer
2024-11-25 23:59:22,762| ERROR   | Traceback (most recent call last):
2024-11-25 23:59:22,763| ERROR   |   File "/opt/conda/lib/python3.9/site-packages/paramiko/transport.py", line 2369, in _check_banner
2024-11-25 23:59:22,763| ERROR   |     buf = self.packetizer.readline(timeout)
2024-11-25 23:59:22,764| ERROR   |   File "/opt/conda/lib/python3.9/site-packages/paramiko/packet.py", line 395, in readline
2024-11-25 23:59:22,764| ERROR   |     buf += self._read_timeout(timeout)
2024-11-25 23:59:22,765| ERROR   |   File "/opt/conda/lib/python3.9/site-packages/paramiko/packet.py", line 663, in _read_timeout
2024-11-25 23:59:22,765| ERROR   |     x = self.__socket.recv(128)
2024-11-25 23:59:22,766| ERROR   | ConnectionResetError: [Errno 104] Connection reset by peer
2024-11-25 23:59:22,766| ERROR   | 
2024-11-25 23:59:22,766| ERROR   | During handling of the above ex

BaseSSHTunnelForwarderError: Could not establish session to SSH gateway

### Retrieve docDB_record from s3 (for backfilling)

In [16]:
import s3fs
import json

S3_RESULTS_ROOT = "aind-scratch-data/aind-dynamic-foraging-analysis/"
fs = s3fs.S3FileSystem(anon=False)

job_hash = "00005a9cf35abbe6aae4e635411630ae3032b4b513ed417a5859ac028ade4354"

with fs.open(S3_RESULTS_ROOT + job_hash + "/docDB_record.json", "r") as f:
    doc_DB_record = json.load(f)

In [17]:
doc_DB_record

{'nwb_name': '711256_2024-04-11_16-21-21.nwb',
 'analysis_spec': {'analysis_name': 'MLE fitting',
  'analysis_ver': 'first version @ 0.10.0',
  'analysis_libs_to_track_ver': ['aind_dynamic_foraging_models'],
  'analysis_args': {'agent_class': 'ForagerQLearning',
   'agent_kwargs': {'number_of_learning_rate': 2,
    'number_of_forget_rate': 1,
    'choice_kernel': 'full',
    'action_selection': 'epsilon-greedy'},
   'fit_kwargs': {'DE_kwargs': {'polish': True, 'seed': 42, 'workers': 8},
    'k_fold_cross_validation': 10}}},
 'job_hash': '00005a9cf35abbe6aae4e635411630ae3032b4b513ed417a5859ac028ade4354',
 'analysis_datetime': '2024-09-20T02:00:37.904226',
 'analysis_time_spent_in_sec': 497.07877802848816,
 'analysis_libs_to_track_ver': {'aind_dynamic_foraging_models': '0.11.0'},
 'analysis_results': {'fit_settings': {'fit_choice_history': [0.0,
    0.0,
    1.0,
    1.0,
    1.0,
    1.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
